# XGBoost: last 30 minutes, excess return vs country (QRT × ETH, 24 Nov 2026)

Self-contained notebook. It runs **on Kaggle** (attach the competition data; a GPU is optional) and
**locally** from the repo (it finds `data/raw/**/x_train.parquet`).

**Pipeline**
1. **Denoise the price paths:** right-align the 5-minute series, fill gaps, and clip bad ticks at
   median ± 5 robust (MAD) sigmas per row.
2. **Scale-free features:** returns over the last k bars divided by intraday volatility; smoothed
   trend (OLS slope) and EMA deviation instead of noisy point-to-point moves; intraday shape.
3. **Country structure:** the target is demeaned per `DATE_ID × COUNTRY`, so the notebook adds
   beta / idiosyncratic moves against the equal-weight country path (best family in PR #8), and
   demeaned values and ranks within each country-day.
4. **Target and weights:** the metric is weighted accuracy with weight = |Close return|. Binary
   mode fits P(return > 0) with |y| weights, winsorised to cut outlier noise. Regression mode fits
   the return and scores its sign (the best linear model in PR #7).
5. **Validation:** the same 5 date-block folds as the team CV (`competition.toml`). One fold is
   monitored **after every boosting round** (weighted accuracy, accuracy, weighted AUC, and
   weighted logloss or RMSE), with early stopping. Then CV over all folds, a refit on all days,
   and the submission.

**Outputs** (`/kaggle/working` on Kaggle, `runs/efe-claude-1_xgb/` locally): `model_final.json`,
`model_holdout.json`, `history_holdout.csv`, `cv_results.json`, `oof_predictions.parquet`,
`feature_importance.csv`, `config.json` and `submission_xgb.csv` (`ID,Close return`, ±1, all ids).

In [ ]:
# ---- configuration -------------------------------------------------------------------------
import os

QUICK = os.environ.get("QUICK", "0") == "1"   # smoke test: few rounds, no full CV
SEED = 42
N_FOLDS = 5              # date-block folds, identical to the team CV (competition.toml)
VAL_FOLD = 4             # the fold monitored round by round
OBJECTIVE = os.environ.get("OBJECTIVE", "binary")  # "binary": P(y > 0) with |y| weights | "regression": fit y, score sign(y_hat)
CLIP_SIGMA = 5.0         # bar returns beyond median ± CLIP_SIGMA * robust sigma are clipped
WEIGHT_CAP_Q = 0.99      # binary: |y| training weights winsorised at this quantile
USE_STOCK_PRIOR = os.environ.get("USE_STOCK_PRIOR", "0") == "1"  # out-of-fold, smoothed per-stock mean of the target (issue #13)
PRIOR_STRENGTH = 50.0    # shrinkage of the stock prior towards 0, in "rows"
N_ROUNDS = 3000
EARLY_STOP = 200         # rounds without improvement on ES_METRIC
ES_METRIC = "wlogloss" if OBJECTIVE == "binary" else "rmse"
PRINT_EVERY = 1          # print the validation metrics every N rounds (1 = every round)
RUN_CV = True            # also train the other folds (quietly) and report CV mean ± std
FULL_REFIT = True        # final model on all training days with mean best rounds × REFIT_FACTOR
REFIT_FACTOR = 1.1

PARAMS = {
    "eta": 0.02, "max_depth": 4, "min_child_weight": 50, "subsample": 0.7,
    "colsample_bytree": 0.6, "reg_lambda": 10.0, "reg_alpha": 0.0, "max_bin": 256,
    "tree_method": "hist", "seed": SEED,
}
if QUICK:
    N_ROUNDS, EARLY_STOP, RUN_CV, PRINT_EVERY = 40, 15, False, 5

In [ ]:
# ---- imports, paths, device ----------------------------------------------------------------
import json
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

SLUG = "qrt-x-eth-hackathon-24-11-2026"
TARGET = "Close return"
KEYS = ["ID", "DATE_ID", "STOCK_ID", "COUNTRY"]


def find_data_dir() -> Path:
    roots = [Path("/kaggle/input")] + [p / "data" / "raw" for p in [Path.cwd(), *Path.cwd().parents]]
    for root in roots:
        if not root.exists():
            continue
        hits = [h for h in root.rglob("x_train.parquet") if (h.parent / "y_train.csv").exists()]
        if hits:
            return sorted(hits, key=lambda h: (SLUG not in str(h), len(h.parts)))[0].parent
    raise FileNotFoundError("x_train.parquet + y_train.csv not found (attach the competition data)")


def find_out_dir(data_dir: Path) -> Path:
    if os.environ.get("OUT_DIR"):
        return Path(os.environ["OUT_DIR"])
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    repo = next((p for p in data_dir.parents if (p / "pyproject.toml").exists()), Path.cwd())
    return repo / "runs" / "efe-claude-1_xgb"     # runs/ is gitignored


DATA_DIR = find_data_dir()
OUT_DIR = find_out_dir(DATA_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
PARAMS["device"] = "cuda" if shutil.which("nvidia-smi") else "cpu"
print("xgboost", xgb.__version__, "| device", PARAMS["device"])
print("data:", DATA_DIR, "\nout: ", OUT_DIR)

In [ ]:
# ---- load ----------------------------------------------------------------------------------
x_train = pd.read_parquet(DATA_DIR / "x_train.parquet")
x_test = pd.read_parquet(DATA_DIR / "x_test.parquet")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")

raw = pd.concat([x_train.assign(is_train=True), x_test.assign(is_train=False)], ignore_index=True)
raw = raw.merge(y_train, on="ID", how="left")
print(f"train {len(x_train):,} rows / {x_train.DATE_ID.nunique()} days | "
      f"test {len(x_test):,} rows / {x_test.DATE_ID.nunique()} days | stocks {raw.STOCK_ID.nunique()}")
print("bars per row:", raw["Price TS"].map(len).value_counts().to_dict())
del x_train, x_test

## 1. Denoise the 5-minute price paths

Rows have 96 bars (48 on half days). Paths are right-aligned so the last column is always the
cutoff. Missing or non-positive prices are filled inside the path. Each bar's log return is then
clipped at median ± `CLIP_SIGMA` robust sigmas of *its own row*, which removes bad prints without
touching normal moves. The raw last bar is kept separately, because the last-bar reversal is the
host's benchmark signal.

In [ ]:
WIDTH = 96
EPS = 1e-12


def price_matrix(series: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    out = np.full((len(series), WIDTH), np.nan)
    n_bars = np.zeros(len(series), dtype=int)
    for i, arr in enumerate(series.to_numpy()):
        arr = np.asarray(arr, dtype=float)[-WIDTH:]
        out[i, WIDTH - len(arr):] = arr
        n_bars[i] = len(arr)
    out[out <= 0] = np.nan
    pad = np.arange(WIDTH)[None, :] < (WIDTH - n_bars)[:, None]
    filled = pd.DataFrame(out).ffill(axis=1).bfill(axis=1).to_numpy()
    filled[pad] = np.nan                      # keep the padding of short days empty
    return filled, n_bars


def clean_bars(prices: np.ndarray, clip_sigma: float) -> tuple[np.ndarray, np.ndarray]:
    raw_r = np.diff(np.log(prices), axis=1)
    med = np.nanmedian(raw_r, axis=1, keepdims=True)
    mad = 1.4826 * np.nanmedian(np.abs(raw_r - med), axis=1, keepdims=True)
    sigma = np.where(mad > 0, mad, np.nanstd(raw_r, axis=1, keepdims=True))
    sigma = np.where(sigma > 0, sigma, 1e-6)
    return raw_r, np.clip(raw_r, med - clip_sigma * sigma, med + clip_sigma * sigma)


t0 = time.time()
PRICES, N_BARS = price_matrix(raw["Price TS"])
RAW_R, R = clean_bars(PRICES, CLIP_SIGMA)
clipped = np.nanmean(RAW_R != R)
print(f"bars clipped: {clipped:.3%} | NaN bars: {np.isnan(R).mean():.3%} | {time.time() - t0:.1f}s")

## 2. Features

* **Per row:** volatility, vol-scaled returns over the last k bars (`z_k`), smoothed trend slopes,
  EMA deviation, lag-1 autocorrelation, intraday shape (range position, drawdown, run-up), and
  liquidity (share of zero bars).
* **Against the country:** beta and correlation to the equal-weight country path, the
  idiosyncratic return over the day and the last k bars, and idiosyncratic volatility.
* **Cross-section:** every key feature demeaned and ranked within its `DATE_ID × COUNTRY`, plus
  day context (country move, dispersion, mean volatility).
* **Stock context, no target:** today's volatility and short-term move relative to the stock's
  own typical values. Train and test rows both have them, and they use no target.

`DATE_ID` and `STOCK_ID` are never features: the date ids are shuffled, and using `STOCK_ID` as
a categorical hurt CV in PR #7.

In [ ]:
LAGS = (1, 2, 3, 6, 12, 24, 48)


def slope(path: np.ndarray, k: int) -> np.ndarray:
    y = path[:, -k:]
    x = np.arange(k) - (k - 1) / 2
    return ((y - y.mean(axis=1, keepdims=True)) * x).sum(axis=1) / (x ** 2).sum()


def ema_last(path: np.ndarray, span: int) -> np.ndarray:
    alpha, ema = 2 / (span + 1), path[:, 0].copy()
    for j in range(1, path.shape[1]):
        ema = alpha * path[:, j] + (1 - alpha) * ema
    return ema


def row_features(r: np.ndarray, raw_r: np.ndarray, n_bars: np.ndarray) -> dict:
    vol = np.nanstd(r, axis=1) + EPS
    path = np.nancumsum(r, axis=1)            # cleaned log-price path, starts at 0
    last = path[:, -1]
    f = {"n_bars": n_bars, "vol": vol, "zero_frac": np.nanmean(raw_r == 0, axis=1),
         "vol_last12": np.nanstd(r[:, -12:], axis=1)}
    f["vol_ratio"] = f["vol_last12"] / vol
    for k in LAGS:
        f[f"ret_{k}"] = np.nansum(r[:, -k:], axis=1)
        f[f"z_{k}"] = f[f"ret_{k}"] / (vol * np.sqrt(k))
    f["ret_day"] = last
    f["z_day"] = last / (vol * np.sqrt(np.maximum(n_bars - 1, 1)))
    f["last_bar_z"] = np.nan_to_num(raw_r[:, -1]) / vol
    for k in (12, 24, 48):
        f[f"slope_{k}"] = slope(path, k) / vol
    for span in (6, 24):
        f[f"ema_dev_{span}"] = (last - ema_last(path, span)) / vol
    tail = r[:, -48:] - np.nanmean(r[:, -48:], axis=1, keepdims=True)
    f["ac1"] = np.nansum(tail[:, 1:] * tail[:, :-1], axis=1) / (np.nansum(tail ** 2, axis=1) + EPS)
    hi, lo = path.max(axis=1), path.min(axis=1)
    f["range_pos"] = (last - lo) / (hi - lo + EPS)
    f["drawdown"] = (last - hi) / vol
    f["runup"] = (last - lo) / vol
    return f


def country_features(r: np.ndarray, keys: pd.DataFrame, n_bars: np.ndarray) -> dict:
    groups = [keys["DATE_ID"].to_numpy(), keys["COUNTRY"].to_numpy()]
    m = pd.DataFrame(r).groupby(groups).transform("mean").to_numpy()   # equal-weight country bars
    rc = r - np.nanmean(r, axis=1, keepdims=True)
    mc = m - np.nanmean(m, axis=1, keepdims=True)
    cov, var_m = np.nanmean(rc * mc, axis=1), np.nanmean(mc ** 2, axis=1) + EPS
    beta = cov / var_m
    idio = r - beta[:, None] * m
    idio_vol = np.nanstd(idio, axis=1) + EPS
    f = {"beta": beta, "corr_cty": cov / np.sqrt(var_m * (np.nanmean(rc ** 2, axis=1) + EPS)),
         "idio_vol": idio_vol, "idio_day": np.nansum(idio, axis=1)}
    f["idio_z_day"] = f["idio_day"] / (idio_vol * np.sqrt(np.maximum(n_bars - 1, 1)))
    for k in (1, 3, 6, 12):
        f[f"idio_z_{k}"] = np.nansum(idio[:, -k:], axis=1) / (idio_vol * np.sqrt(k))
    return f


CS_COLS = ["z_1", "z_3", "z_6", "z_12", "z_48", "z_day", "last_bar_z", "slope_12", "slope_48",
           "ema_dev_6", "ema_dev_24", "vol", "vol_ratio", "range_pos", "idio_z_day", "idio_z_6"]


def cross_section(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["DATE_ID", "COUNTRY"])
    rel = (df[CS_COLS] - g[CS_COLS].transform("mean")).add_suffix("_rel")
    rank = g[CS_COLS].rank(pct=True).add_suffix("_rank")
    ctx = pd.DataFrame({
        "cty_ret_day": g["ret_day"].transform("mean"), "cty_ret_6": g["ret_6"].transform("mean"),
        "cty_disp_day": g["ret_day"].transform("std"), "cty_vol": g["vol"].transform("mean"),
        "cty_n": g["ret_day"].transform("size")}, index=df.index)
    s = df.groupby("STOCK_ID")
    stock = pd.DataFrame({
        "vol_vs_stock": df["vol"] / s["vol"].transform("median"),
        "z6_vs_stock": df["z_6"] - s["z_6"].transform("mean"),
        "last_bar_vs_stock": df["last_bar_z"] - s["last_bar_z"].transform("mean")}, index=df.index)
    return pd.concat([rel, rank, ctx, stock], axis=1)


t0 = time.time()
feats = pd.DataFrame({**row_features(R, RAW_R, N_BARS), **country_features(R, raw[KEYS], N_BARS)})
df = pd.concat([raw[KEYS + ["is_train", TARGET]], feats], axis=1)
df["country_code"] = df["COUNTRY"].astype("category").cat.codes
df["open_min"] = raw["OPEN_TIME"].str.slice(0, 2).astype(int) * 60 + raw["OPEN_TIME"].str.slice(3, 5).astype(int)
df = pd.concat([df, cross_section(df)], axis=1)
df = df.replace([np.inf, -np.inf], np.nan)
FEATURES = [c for c in df.columns if c not in KEYS + ["is_train", TARGET]]
print(f"{len(FEATURES)} features in {time.time() - t0:.1f}s")
del PRICES, RAW_R, R, feats

## 3. Target, weights, folds, optional stock prior

The folds are contiguous blocks of the sorted unique `DATE_ID`s, the same as
`qrtc.cv.date_block_splits` with 5 folds and no embargo. The ids are shuffled, so a block is a
random set of whole days, which matches how the test days differ from the training days.

In [ ]:
train_df = df[df["is_train"]].reset_index(drop=True)
Y = train_df[TARGET].to_numpy()
DATES = train_df["DATE_ID"].to_numpy()
print(f"positive share {np.mean(Y > 0):.4f} | |y|-weighted positive share "
      f"{np.abs(Y)[Y > 0].sum() / np.abs(Y).sum():.4f}")


def date_block_folds(dates: np.ndarray, n_splits: int) -> list[tuple[np.ndarray, np.ndarray]]:
    uniq = np.sort(pd.unique(dates))
    pos = np.searchsorted(uniq, dates)
    folds = []
    for block in np.array_split(np.arange(len(uniq)), n_splits):
        val = (pos >= block[0]) & (pos <= block[-1])
        folds.append((np.flatnonzero(~val), np.flatnonzero(val)))
    return folds


def fit_target(y: np.ndarray) -> tuple[np.ndarray, np.ndarray | None]:
    if OBJECTIVE == "binary":
        w = np.minimum(np.abs(y), np.quantile(np.abs(y), WEIGHT_CAP_Q))
        return (y > 0).astype(float), w / w.mean()
    return y / Y.std(), None


def stock_prior(frame: pd.DataFrame, fit_idx: np.ndarray) -> np.ndarray:
    # Smoothed sum(y) / sum|y| per stock: out-of-fold inside fit_idx, full fit_idx for other rows.
    stock, y = frame["STOCK_ID"].to_numpy(), frame[TARGET].to_numpy()

    def prior_map(idx):
        d = pd.DataFrame({"s": stock[idx], "y": y[idx], "a": np.abs(y[idx])}).groupby("s").sum()
        return d["y"] / (d["a"] + PRIOR_STRENGTH * np.abs(y[idx]).mean())

    out = pd.Series(stock).map(prior_map(fit_idx)).to_numpy(dtype=float)
    for tr, va in date_block_folds(frame["DATE_ID"].to_numpy()[fit_idx], N_FOLDS):
        out[fit_idx[va]] = pd.Series(stock[fit_idx[va]]).map(prior_map(fit_idx[tr])).to_numpy()
    return np.nan_to_num(out)


def design(frame: pd.DataFrame, fit_idx: np.ndarray) -> pd.DataFrame:
    X = frame[FEATURES].astype(np.float32)
    if USE_STOCK_PRIOR:
        X = X.assign(stock_prior=stock_prior(frame, fit_idx).astype(np.float32))
    return X


FOLDS = date_block_folds(DATES, N_FOLDS)
print("fold sizes (train / val rows):", [(len(a), len(b)) for a, b in FOLDS])

## 4. Metrics after every boosting round

`weighted_accuracy` = Σ |y| · [sign(pred) = sign(y)] / Σ |y|, which is the competition metric. The
callback prints it for the monitored fold after each round, together with accuracy, |y|-weighted
AUC, and weighted logloss (binary) or RMSE (regression). Every round is stored in
`history_holdout.csv`.

In [ ]:
THRESHOLD = 0.5 if OBJECTIVE == "binary" else 0.0


def metrics(score: np.ndarray, y: np.ndarray) -> dict:
    w, pos = np.abs(y), y > 0
    correct = (score > THRESHOLD) == pos
    out = {"wacc": float((w * correct).sum() / w.sum()), "acc": float(correct.mean()),
           "wauc": float(roc_auc_score(pos, score, sample_weight=w))}
    if OBJECTIVE == "binary":
        p = np.clip(score, 1e-7, 1 - 1e-7)
        out["wlogloss"] = float(-(w * np.where(pos, np.log(p), np.log(1 - p))).sum() / w.sum())
    else:
        out["rmse"] = float(np.sqrt(np.mean((score - y / Y.std()) ** 2)))
    return out


SIGNED_Y: dict[int, np.ndarray] = {}         # id(DMatrix) -> true signed target


def feval(predt: np.ndarray, dmat: xgb.DMatrix) -> list[tuple[str, float]]:
    # With a built-in objective XGBoost passes transformed predictions (probabilities).
    m = metrics(predt, SIGNED_Y[id(dmat)])
    order = [k for k in m if k != ES_METRIC] + [ES_METRIC]    # early stopping uses ES_METRIC
    return [(k, m[k]) for k in order]


class RoundLogger(xgb.callback.TrainingCallback):
    def __init__(self, every: int, verbose: bool = True):
        self.every, self.verbose, self.rows, self.t0 = every, verbose, [], time.time()

    def after_iteration(self, model, epoch, evals_log) -> bool:
        row = {"round": epoch + 1}
        for data, ms in evals_log.items():
            row.update({f"{data}_{k}": v[-1] for k, v in ms.items()})
        self.rows.append(row)
        if self.verbose and (epoch == 0 or (epoch + 1) % self.every == 0):
            val = "  ".join(f"{k.removeprefix('valid_')} {v:.5f}" for k, v in row.items()
                            if k.startswith("valid_"))
            print(f"[{epoch + 1:5d}] {val}  | train wacc {row.get('train_wacc', np.nan):.5f}"
                  f"  ({time.time() - self.t0:.0f}s)")
        return False


def dmatrix(X: pd.DataFrame, y: np.ndarray, with_fit_weights: bool) -> xgb.DMatrix:
    label, w = fit_target(y)
    d = xgb.DMatrix(X, label=label, weight=w if with_fit_weights else None)
    SIGNED_Y[id(d)] = y
    return d


def train_fold(tr: np.ndarray, va: np.ndarray, verbose: bool) -> tuple[xgb.Booster, pd.DataFrame, np.ndarray]:
    X = design(train_df, tr)
    dtr, dva = dmatrix(X.iloc[tr], Y[tr], True), dmatrix(X.iloc[va], Y[va], False)
    params = {**PARAMS, "objective": "binary:logistic" if OBJECTIVE == "binary" else "reg:squarederror",
              "disable_default_eval_metric": 1}
    logger = RoundLogger(PRINT_EVERY, verbose)
    stop = xgb.callback.EarlyStopping(rounds=EARLY_STOP, metric_name=ES_METRIC, data_name="valid",
                                      maximize=ES_METRIC in ("wacc", "acc", "wauc"), save_best=True)
    booster = xgb.train(params, dtr, num_boost_round=N_ROUNDS, evals=[(dtr, "train"), (dva, "valid")],
                        custom_metric=feval, callbacks=[logger, stop], verbose_eval=False)
    pred = booster.predict(dva)
    return booster, pd.DataFrame(logger.rows), pred

## 5. Train the monitored fold (metrics every round)

In [ ]:
tr, va = FOLDS[VAL_FOLD]
holdout_model, history, holdout_pred = train_fold(tr, va, verbose=True)
best_round = holdout_model.best_iteration + 1
holdout = metrics(holdout_pred, Y[va])
history.to_csv(OUT_DIR / "history_holdout.csv", index=False)
holdout_model.save_model(OUT_DIR / "model_holdout.json")

bench = -train_df["last_bar_z"].to_numpy()[va]        # host benchmark: -(last 5-minute return)
w_va = np.abs(Y[va])
bench_wacc = (w_va * ((bench > 0) == (Y[va] > 0))).sum() / w_va.sum()
print(f"\nbest round {best_round} | holdout {json.dumps({k: round(v, 5) for k, v in holdout.items()})}")
print(f"benchmark -(last bar) on the same days: wacc {bench_wacc:.5f}")

In [ ]:
# ---- learning curves -----------------------------------------------------------------------
import matplotlib.pyplot as plt

second = ES_METRIC
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["wacc", second]):
    for split in ("train", "valid"):
        ax.plot(history["round"], history[f"{split}_{metric}"], label=split)
    ax.axvline(best_round, color="grey", ls="--", lw=1, label="best round")
    ax.set_title(metric)
    ax.set_xlabel("boosting round")
    ax.legend()
axes[0].axhline(bench_wacc, color="red", ls=":", lw=1, label="benchmark")
plt.tight_layout()
plt.show()

In [ ]:
# ---- calm vs volatile days (test days are calmer, see the data audit in PR #15) ---------------
day_vol = train_df.iloc[va].groupby("DATE_ID")["vol"].transform("mean").to_numpy()
calm = day_vol <= np.median(day_vol)
for name, mask in (("calm days", calm), ("volatile days", ~calm)):
    print(f"{name:14s} wacc {metrics(holdout_pred[mask], Y[va][mask])['wacc']:.5f}")

## 6. Cross-validation over all 5 folds (comparable with the team leaderboard)

In [ ]:
oof = np.full(len(train_df), np.nan)
oof[va] = holdout_pred
fold_scores, best_rounds = {VAL_FOLD: holdout["wacc"]}, {VAL_FOLD: best_round}
if RUN_CV:
    for k, (tr_k, va_k) in enumerate(FOLDS):
        if k == VAL_FOLD:
            continue
        t0 = time.time()
        model_k, _, pred_k = train_fold(tr_k, va_k, verbose=False)
        oof[va_k] = pred_k
        fold_scores[k], best_rounds[k] = metrics(pred_k, Y[va_k])["wacc"], model_k.best_iteration + 1
        print(f"fold {k}: wacc {fold_scores[k]:.5f} | best round {best_rounds[k]} | {time.time() - t0:.0f}s")

scores = np.array([fold_scores[k] for k in sorted(fold_scores)])
cv = {"objective": OBJECTIVE, "fold_wacc": fold_scores, "best_rounds": best_rounds,
      "cv_wacc_mean": float(scores.mean()), "cv_wacc_std": float(scores.std()),
      "pooled_oof": metrics(oof[~np.isnan(oof)], Y[~np.isnan(oof)])}
print(f"\nweighted accuracy: {cv['cv_wacc_mean']:.5f} ± {cv['cv_wacc_std']:.5f} over {len(scores)} fold(s)"
      "   (team baseline 1124-184854_dmr-claude-0_baseline-lgbm: 0.53289 ± 0.00575)")
(OUT_DIR / "cv_results.json").write_text(json.dumps(cv, indent=2, default=str))
pd.DataFrame({"ID": train_df["ID"], "DATE_ID": DATES, "y": Y, "oof": oof}).to_parquet(
    OUT_DIR / "oof_predictions.parquet", index=False)

## 7. Refit on all training days, save the model, write the submission

Kaggle expects **every** id (train and test, 219,157 rows) with `Close return` = ±1. An exact 0 is
always wrong, so a prediction on the threshold falls back to the benchmark's sign.

In [ ]:
all_idx = np.arange(len(train_df))
if FULL_REFIT:
    n_final = max(1, int(np.mean(list(best_rounds.values())) * REFIT_FACTOR))
    X_all = design(train_df, all_idx)
    d_all = dmatrix(X_all, Y, True)
    params = {**PARAMS, "objective": "binary:logistic" if OBJECTIVE == "binary" else "reg:squarederror"}
    final_model = xgb.train(params, d_all, num_boost_round=n_final)
    print(f"refit on {len(train_df):,} rows for {n_final} rounds")
else:
    final_model = holdout_model
final_model.save_model(OUT_DIR / "model_final.json")

# Features for every id; the stock prior (if on) uses all training rows.
fit_rows = np.flatnonzero(df["is_train"].to_numpy())
X_sub = design(df, fit_rows)
score = final_model.predict(xgb.DMatrix(X_sub))
fallback = np.where(-df["last_bar_z"].to_numpy() > 0, 1, -1)
signs = np.where(score > THRESHOLD, 1, np.where(score < THRESHOLD, -1, fallback))
sub = pd.DataFrame({"ID": df["ID"].to_numpy(), TARGET: signs}).sort_values("ID")
assert len(sub) == len(df) and sub["ID"].is_unique and set(sub[TARGET]) <= {-1, 1}
sub.to_csv(OUT_DIR / "submission_xgb.csv", index=False)
test_share = (signs[~df["is_train"].to_numpy()] > 0).mean()
print(f"submission: {len(sub):,} rows, +1 share on test rows {test_share:.3f} -> {OUT_DIR / 'submission_xgb.csv'}")

In [ ]:
# ---- feature importance and run config -----------------------------------------------------
gain = pd.Series(final_model.get_score(importance_type="total_gain")).sort_values(ascending=False)
gain.rename("total_gain").to_csv(OUT_DIR / "feature_importance.csv")
print(gain.head(25).round(2).to_string())

config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool, dict))
          and k not in ("SIGNED_Y",)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2, default=str))
print("\nsaved:", sorted(p.name for p in OUT_DIR.iterdir()))